# jaxfne — Étude No. 9 · Simple Local Oddball Task

Standard/deviant local oddball paradigm and the local mismatch response: a
repetitive standard stimulus with an occasional deviant, measuring the
change in L4 excitatory firing rate right after the deviant vs. the matched
position in an all-standard control sequence.


## 1. Setup

In [1]:
import numpy as np
import jax.numpy as jnp
import jaxfne as jtfne

print("jaxfne", jtfne.__version__)


jaxfne 0.4.4


## 2. Config

In [2]:
N_NEURONS = 200
SEED = 0
DT_MS = 0.5
FX_MS = 200.0        # fixation before the first stimulus
P_DUR_MS = 100.0      # stimulus duration
ISI_MS = 300.0        # inter-stimulus interval
N_STIM = 5            # stimuli per trial
TOTAL_MS = FX_MS + N_STIM * ISI_MS
N_TRIALS = 10         # trials per condition
DEVIANT_POSITION = 3  # 0-indexed slot that carries the deviant

cfg = (
    jtfne.build_laminar_column(name="V1", n=N_NEURONS, ei_profile="canonical")
    .runtime(seed=SEED, recurrent_backend="edge_list")
    .set_emitter("izhikevich", "cortical_eig")
    .probes(["spikes", "V_m"], n_contacts=8)
    .field(domain="laminar_column", conductivity="proxy", boundary="mean_zero_neumann")
)
model = jtfne.construct(cfg)
nt = model.neuron_table()
l4e_idx = [i for i, r in enumerate(nt) if r.get("layer") == "L4" and r.get("cell_type") == "E"]
print(f"Column: {len(nt)} neurons, L4 E (stimulus targets): {len(l4e_idx)}")


Column: 200 neurons, L4 E (stimulus targets): 14


## 3. Stimulus schedules

Two conditions built from the same slot structure: `standard` (all 5 stimuli
identical) and `local_oddball` (slot `DEVIANT_POSITION` carries a higher-
amplitude deviant). Both schedules must be wrapped in
`jaxfne.StimulusSchedule` — a bare list of event dicts is silently ignored by
`simulate()` (confirmed 2026-07-01; `Model._resolve_stimulus_schedule` only
recognizes `StimulusSchedule`/`ParadigmCondition`, anything else resolves to
no stimulus with no error or warning).

In [3]:
def make_schedule(deviant_position):
    events = []
    for k in range(N_STIM):
        onset = FX_MS + k * ISI_MS
        is_deviant = (k == deviant_position)
        events.append({
            "onset_ms": onset, "duration_ms": P_DUR_MS,
            "amplitude": 8.0 if is_deviant else 4.0,
            "label": "deviant" if is_deviant else "standard",
            "is_drive_event": True,
            "target_indices": l4e_idx,
        })
    return jtfne.StimulusSchedule(events=events, n_neurons=len(nt))

sched_standard = make_schedule(deviant_position=-1)       # all standard
sched_local_oddball = make_schedule(deviant_position=DEVIANT_POSITION)


## 4. Run

In [4]:
def mean_rate_post_deviant(sched, n_trials):
    rates = []
    window_start = int((FX_MS + DEVIANT_POSITION * ISI_MS) / DT_MS)
    window_end = int((FX_MS + DEVIANT_POSITION * ISI_MS + P_DUR_MS + 100.0) / DT_MS)
    for i in range(n_trials):
        sig = jtfne.simulate(
            model, sim=jtfne.Simulation(duration_ms=TOTAL_MS, dt_ms=DT_MS, seed=SEED + i),
            paradigm=sched,
        )
        assert bool(jnp.all(jnp.isfinite(sig.V_m))), "non-finite V_m -- do not trust this run"
        spk = np.asarray(sig.spikes)
        rates.append(spk[window_start:window_end, l4e_idx].mean() * 1000.0 / DT_MS)
    return np.array(rates)

rates_std = mean_rate_post_deviant(sched_standard, N_TRIALS)
rates_odd = mean_rate_post_deviant(sched_local_oddball, N_TRIALS)
print(f"standard: {rates_std.mean():.2f} +- {rates_std.std():.2f} Hz  ({N_TRIALS} trials)")
print(f"oddball:  {rates_odd.mean():.2f} +- {rates_odd.std():.2f} Hz  ({N_TRIALS} trials)")


standard: 12.00 +- 0.56 Hz  (10 trials)
oddball:  13.46 +- 0.48 Hz  (10 trials)


## 5. Objective — local mismatch response

In [5]:
mismatch_hz = float(rates_odd.mean() - rates_std.mean())
print(f"local mismatch response (oddball - standard): {mismatch_hz:+.2f} Hz")

# Computational diagnostic, not a claim of biological mismatch negativity --
# this is a same-model firing-rate comparison, not a validated ERP/MMN result.
assert np.isfinite(mismatch_hz)


local mismatch response (oddball - standard): +1.46 Hz


## 6. Export

In [6]:
import json as _json
from pathlib import Path

OUT_DIR = Path("local/etude9")
OUT_DIR.mkdir(parents=True, exist_ok=True)

manifest = {
    "notebook": "jaxfne_etude_no_9_local_oddball",
    "jaxfne_version": jtfne.__version__,
    "paradigm": {
        "name": "local_oddball", "n_stim": N_STIM, "deviant_position": DEVIANT_POSITION,
        "p_dur_ms": P_DUR_MS, "isi_ms": ISI_MS, "fixation_ms": FX_MS,
        "stimulus_target": "L4_E_neurons", "n_stimulus_targets": len(l4e_idx),
    },
    "config": {"N_NEURONS": N_NEURONS, "N_TRIALS": N_TRIALS, "DT_MS": DT_MS, "SEED": SEED},
    "results": {
        "standard_rate_hz_mean": float(rates_std.mean()), "standard_rate_hz_std": float(rates_std.std()),
        "oddball_rate_hz_mean": float(rates_odd.mean()), "oddball_rate_hz_std": float(rates_odd.std()),
        "mismatch_response_hz": mismatch_hz,
    },
    "claim_level": "computational_scaffold",
    "physical_amplitude_calibrated": False,
}
(OUT_DIR / "manifest.json").write_text(_json.dumps(manifest, indent=2))
print("wrote", OUT_DIR / "manifest.json")


wrote local/etude9/manifest.json
